# 03 · Columns & built-in functions

`pyspark.sql.functions` (imported as `F`) has **hundreds** of functions for maths, strings, dates, nulls,
conditionals… They run inside the JVM, so they're fast. **Rule #1 of PySpark: if a built-in function exists, use it.**

Full list: <https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html>

In [ ]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [ ]:
customers = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)
orders.show(3)

## Literals, arithmetic, aliases and casting

In [ ]:
orders.select(
    "order_id",
    "quantity",
    (F.col("quantity") * 2).alias("double_qty"),
    F.lit("web").alias("channel"),                        # a constant value
    F.col("quantity").cast("string").alias("qty_str"),    # change type
    F.round(F.col("quantity") / 3, 2).alias("third"),
).show(3)

### Spark 4 is strict about bad casts (ANSI mode)
Since Spark 4.0, *ANSI mode* is on by default: casting `"N/A"` to an int **raises an error** instead of
silently returning null. When the data might be dirty, use `try_cast` (and friends like `try_divide`,
`try_to_timestamp`) which return `null` on failure.

In [ ]:
bad = spark.createDataFrame([("12",), ("N/A",), ("7",)], "raw string")
try:
    bad.select(F.col("raw").cast("int")).show()
except Exception as e:
    print("cast failed:", type(e).__name__)

bad.select("raw", F.col("raw").try_cast("int").alias("as_int")).show()

## Conditional logic: `when` / `otherwise`
Spark's version of `if / elif / else` (or SQL's `CASE WHEN`). Conditions are checked in order; the first match wins.
Without `.otherwise(...)` non-matching rows get `null`.

In [ ]:
orders.select(
    "order_id", "quantity", "status",
    F.when(F.col("status") == "completed", "✓")
     .when(F.col("status") == "returned", "↩")
     .otherwise("✗").alias("icon"),
).show(5)

## String functions

In [ ]:
customers.select(
    "name",
    F.upper("name").alias("upper"),
    F.length("name").alias("len"),
    F.split("name", " ").getItem(0).alias("first_name"),
    F.substring("name", 1, 3).alias("first3"),                     # 1-based!
    F.concat_ws(" | ", "name", "country").alias("joined"),
    F.regexp_replace("email", r"\d+", "#").alias("no_digits"),
    F.regexp_extract("email", r"^([a-z]+)\.", 1).alias("email_first"),
).show(5, truncate=False)

In [ ]:
customers.filter(F.col("email").endswith("@company.io")).select("name", "email").show(3)
customers.filter(F.col("name").like("%Kim%")).select("name").show(3)       # SQL LIKE: % = anything
customers.filter(F.col("name").rlike("^(Alice|Bob) ")).select("name").show(3)  # regex

## Working with nulls
A very handy idiom: count the nulls in every column in one pass.

In [ ]:
customers.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in customers.columns]).show()

In [ ]:
customers.fillna({"country": "Unknown"}).filter(F.col("country") == "Unknown").show(3)   # replace nulls
print(customers.dropna(subset=["country"]).count(), "rows with a country")              # drop null rows
customers.select("name", F.coalesce("country", F.lit("??")).alias("country")).show(3)  # first non-null

## Dates and timestamps
`order_ts` was inferred as a `timestamp`. Common functions:

In [ ]:
orders.select(
    "order_ts",
    F.to_date("order_ts").alias("date"),
    F.year("order_ts").alias("year"),
    F.month("order_ts").alias("month"),
    F.hour("order_ts").alias("hour"),
    F.dayofweek("order_ts").alias("dow"),                 # 1 = Sunday … 7 = Saturday
    F.date_format("order_ts", "EEEE").alias("day_name"),
    F.date_trunc("month", "order_ts").alias("month_start"),
    F.datediff(F.lit("2025-01-01"), "order_ts").alias("days_to_2025"),
    F.date_add(F.to_date("order_ts"), 30).alias("plus_30d"),
).show(3, truncate=False)

Parsing strings in non-standard formats — pass a [datetime pattern](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html):

In [ ]:
spark.createDataFrame([("25/12/2024",), ("01/02/2025",)], "d string") \
     .select("d", F.to_date("d", "dd/MM/yyyy").alias("parsed")).show()

## Applying a function to many columns
`select` accepts a Python list, so list comprehensions are a superpower:

In [ ]:
customers.select([F.upper(c).alias(c) if t == "string" else F.col(c) for c, t in customers.dtypes]).show(3)

### Exercise 1
Create `with_domain`: `customers` plus a column **`domain`** containing the part of the email **after the `@`** (e.g. `gmail.com`).

*Hint:* `F.split(...).getItem(1)` or `F.regexp_extract` or `F.substring_index`.

In [ ]:
with_domain = None  # TODO

In [ ]:
# Check — run me after your answer
assert "domain" in with_domain.columns
assert with_domain.filter(F.col("domain") == "gmail.com").count() == 48
print("Correct!")

### Exercise 2
Create `with_year`: `customers` plus an integer column **`signup_year`**.

In [ ]:
with_year = None  # TODO

In [ ]:
# Check — run me after your answer
assert dict(with_year.dtypes)["signup_year"] == "int"
assert with_year.filter(F.col("signup_year") == 2024).count() == 107
print("Correct!")

### Exercise 3
Create `clean_country`: `customers` where null countries are replaced by **`"Unknown"`**.

In [ ]:
clean_country = None  # TODO

In [ ]:
# Check — run me after your answer
assert clean_country.filter(F.col("country").isNull()).count() == 0
assert clean_country.filter(F.col("country") == "Unknown").count() == 15
print("Correct!")

### Exercise 4
Create `sized`: `orders` plus a column **`size`**: `"small"` when quantity is 1, `"medium"` for 2–3, `"large"` for 4 or more.

In [ ]:
sized = None  # TODO

In [ ]:
# Check — run me after your answer
got = {r["size"]: r["count"] for r in sized.groupBy("size").count().collect()}
assert got == {'small': 2034, 'medium': 1990, 'large': 976}, got
print("Correct!")

### Exercise 5
Store in `weekend_orders` the number of orders placed on a **Saturday or Sunday**.

In [ ]:
weekend_orders = None  # TODO

In [ ]:
# Check — run me after your answer
assert weekend_orders == 1371, weekend_orders
print("Correct!")

---
Stuck? Worked answers (with outputs) are in [`../solutions/03_columns_and_functions.ipynb`](../solutions/03_columns_and_functions.ipynb).